# EXP-STIG-001 evidence audit
Observer-only companion for the fresh R2 execution. Run only after result.json exists. The original V1 execution is censored, not a scientific comparison; its independent record is EXP_STIG_001_V1/censored-audit.json. This notebook does not call models, train, or change simulation state. It independently checks cohort completeness, assignment, denominators, source availability, and hashes before invoking the deterministic replay auditor. Zero reuse is not proof of equivalence; general task utility and durable specialization are not identifiable in this bounded assay.


In [ ]:
from pathlib import Path
import json, hashlib, sqlite3, collections, subprocess, sys
EVIDENCE = Path('E:/WORLD/AI_CACHE/evidence/EXP_STIG_001_R2')
REPOSITORY = Path('E:/WORLD/A-LifeLM_MD_CANONICAL_v1')
def canonical(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(',', ':'), allow_nan=False).encode('utf-8')
def sha(data):
    return hashlib.sha256(data).hexdigest()
def read(name):
    return json.loads((EVIDENCE/name).read_bytes())
def check_manifest(value):
    assert value['manifest_digest'] == sha(canonical({k:v for k,v in value.items() if k != 'manifest_digest'}))
result, planned, archive = read('result.json'), read('planned.json'), read('source-archive.json')
for manifest in (result, planned, archive):
    check_manifest(manifest)
assert result['status'] == 'COMPLETE'
assert result['planned_digest'] == planned['manifest_digest']
assert archive['sources'] == planned['sources']
for filename, expected in result['evidence_hashes'].items():
    assert sha((EVIDENCE/filename).read_bytes()) == expected, filename
for filename, expected in archive['sources'].items():
    assert sha((EVIDENCE/'cas'/expected[:2]/expected).read_bytes()) == expected, filename
print('Manifest, evidence-file and source-CAS integrity: PASS')


In [ ]:
db = sqlite3.connect((EVIDENCE/'world.sqlite').as_uri() + '?mode=ro', uri=True)
db.row_factory = sqlite3.Row
runs = db.execute('SELECT id, initial FROM stig_runs ORDER BY id').fetchall()
commits = db.execute('SELECT * FROM stig_commits ORDER BY run,tick').fetchall()
assert len(runs) == 9 and len(commits) == 36
assert len({(r['run'],r['tick']) for r in commits}) == 36
summary = {c:collections.Counter() for c in 'ABC'}
opportunities, expected_exposures, seen_exposures = set(), set(), set()
seed_by_key = {}
states = {r['id']:json.loads(r['initial']) for r in runs}
for row in commits:
    condition = row['run'].rsplit('-',1)[1]
    block = int(row['run'].split('-')[-2])
    assignment = planned['config']['model_assignment'][planned['config']['blocks'].index(block)]
    before = states[row['run']]
    request, events = json.loads(row['request']), json.loads(row['events'])
    assert request['before'] == sha(canonical(before)) and before['tick'] == row['tick']
    assert len(request['attempts']) == len(events) == 3
    assert sorted(a['actor'] for a in request['attempts']) == planned['config']['agents']
    for attempt in request['attempts']:
        assert attempt['opportunity'] not in opportunities
        opportunities.add(attempt['opportunity'])
        seed_by_key[(row['run'],row['tick'],attempt['actor'])] = attempt['seed']
        index = planned['config']['agents'].index(attempt['actor'])
        assert attempt['model'] == planned['models'][assignment[index]]['sha256']
        key = (row['run'],attempt['actor'])
        if key not in seen_exposures and any(t['creator'] != attempt['actor'] for t in before['traces']):
            seen_exposures.add(key)
            expected_exposures.add((row['run'],row['tick'],attempt['actor']))
    for event in events:
        summary[condition]['opportunities'] += 1
        summary[condition][event['status']] += 1
        if event['status'] == 'ACCEPTED':
            summary[condition][event['action']] += 1
    states[row['run']] = json.loads(row['state'])
    assert sha(canonical(states[row['run']])) == row['hash']
assert len(opportunities) == 108
assert all(v['opportunities'] == 36 for v in summary.values())
assert all(v['tick'] == 4 for v in states.values())
probes, costs = read('counterfactuals.json'), read('costs.json')
assert {(p['run_id'],p['tick'],p['actor']) for p in probes} == expected_exposures
assert len(probes) == len(expected_exposures)
assert sum(c['phase'] == 'primary' for c in costs) == 108
assert len(costs) == 108 + 2*len(probes) == result['requests']
for cost in costs:
    if cost['phase'] != 'primary':
        continue
    value = cost['response_ref']
    raw_response = (EVIDENCE/'cas'/value[:2]/value).read_bytes()
    assert sha(raw_response) == value
    response = json.loads(raw_response)
    settings = response['generation_settings']
    assert settings['seed'] == seed_by_key[(cost['run_id'],cost['tick'],cost['actor'])]
    for key in ('temperature','top_k','top_p','min_p','repeat_penalty','n_predict','stream'):
        assert settings[key] == planned['config']['generation'][key], key
    assert settings['grammar'] == '' and settings['lora'] == []
    assert settings['ignore_eos'] is False and settings['logit_bias'] == []
    assert settings['stop'] == [] and response['timings']['cache_n'] == 0
    assert response['truncated'] is False
db.close()
print(json.dumps({c:dict(v) for c,v in summary.items()},indent=2))
print('Cohort, model assignment, unique opportunities, exposure selection and request denominators: PASS')


In [ ]:
# No model inference: raw proposals are replayed through deterministic laws.
subprocess.run([sys.executable, str(REPOSITORY/'tools/exp_stig_001.py'), '--audit', '--output', str(EVIDENCE)], cwd=REPOSITORY, check=True)
